In [1]:
!pip install namedivider-python python-romkan-ng

import pandas as pd
import numpy as np
import romkan
import os
import re
import kagglehub
from namedivider import BasicNameDivider
from time import time
from typing import List, Tuple, Dict, Callable
import math
import matplotlib.pyplot as plt
import importlib

IS_KAGGLE = os.path.exists("/kaggle/input")


def get_jpersonwiki_path():
    """(csvパス, Kaggle datasetのバージョン番号)を返す。

    Kaggle上でも/kaggle/inputのマウントは使わず、常にkagglehubでAPI経由で
    最新バージョンを取得する — スケジュール実行時にinputが古いバージョンに
    固定されたまま更新されない問題を避けるため。
    """
    path = kagglehub.dataset_download("rentoda/jpersonwiki")
    m = re.search(r"versions/(\d+)", path)
    version = int(m.group(1)) if m else None
    return os.path.join(path, "jpersonwiki.csv"), version


def get_latest_commit_sha(repo: str, branch: str) -> str:
    """生成に使った外部GitHubリポジトリの、現時点でのブランチ最新commit hashを取得する(再現性メモ用)。"""
    try:
        import requests

        r = requests.get(
            f"https://api.github.com/repos/{repo}/commits/{branch}", timeout=10
        )
        r.raise_for_status()
        return r.json()["sha"]
    except Exception:
        return "unknown"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.3 MB/s eta 0:00:00


In [2]:
import re
import csv


def parse_enamdict(input_path, output_path):
    rows = []

    with open(input_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Extract kanji, hiragana, sex, romaji
            match = re.match(r"^(.+?)\s*\[(.+?)\]\s*/\(([mf])\)\s*(.+?)/", line)
            if not match:
                continue

            kanji, hiragana, sex, romaji = match.groups()
            rows.append(
                {
                    "sex": sex,
                    "kanji": kanji,
                    "hiragana": hiragana,
                    "romaji": romaji.strip(),
                }
            )

    with open(output_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["sex", "kanji", "hiragana", "romaji"])
        writer.writeheader()
        writer.writerows(rows)


def parse_name_origin_to_flat_csv(sex, input_path, output_path):
    rows = []

    with open(input_path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 3:
                continue
            hiragana, romaji, *kanjis = parts
            for kanji in kanjis:
                rows.append(
                    {"kanji": kanji, "hiragana": hiragana, "romaji": romaji, "sex": sex}
                )

    pd.DataFrame(rows).to_csv(output_path, index=False)

In [3]:
# https://github.com/rgamici/japanese-names
!curl --create-dirs -o ./jnamegender/intermidiate/enumdict_male.txt "https://raw.githubusercontent.com/rgamici/japanese-names/refs/heads/master/male"
!curl --create-dirs -o ./jnamegender/intermidiate/enumdict_female.txt "https://raw.githubusercontent.com/rgamici/japanese-names/refs/heads/master/female"
!sync

parse_enamdict("./jnamegender/intermidiate/enumdict_male.txt", "./jnamegender/intermidiate/enumdict_male.csv")
parse_enamdict(
    "./jnamegender/intermidiate/enumdict_female.txt", "./jnamegender/intermidiate/enumdict_female.csv"
)
!sync

enamdict_male = pd.read_csv("./jnamegender/intermidiate/enumdict_male.csv")
enamdict_female = pd.read_csv("./jnamegender/intermidiate/enumdict_female.csv")
enamdict = pd.concat([enamdict_male, enamdict_female])

# https://github.com/shuheilocale/japanese-personal-name-dataset
!curl --create-dirs -o ./jnamegender/intermidiate/name_origin_male.txt "https://raw.githubusercontent.com/shuheilocale/japanese-personal-name-dataset/refs/heads/main/japanese_personal_name_dataset/dataset/first_name_man_org.csv"
!curl --create-dirs -o ./jnamegender/intermidiate/name_origin_female.txt "https://raw.githubusercontent.com/shuheilocale/japanese-personal-name-dataset/refs/heads/main/japanese_personal_name_dataset/dataset/first_name_woman_org.csv"
!sync

parse_name_origin_to_flat_csv(
    "M", "./jnamegender/intermidiate/name_origin_male.txt", "./jnamegender/intermidiate/name_origin_male.csv"
)
parse_name_origin_to_flat_csv(
    "F",
    "./jnamegender/intermidiate/name_origin_female.txt",
    "./jnamegender/intermidiate/name_origin_female.csv",
)
!sync

name_origin_male = pd.read_csv("./jnamegender/intermidiate/name_origin_male.csv")
name_origin_female = pd.read_csv("./jnamegender/intermidiate/name_origin_female.csv")
name_origin = pd.concat([name_origin_male, name_origin_female])

enamdict_source_commit = get_latest_commit_sha("rgamici/japanese-names", "master")
name_origin_source_commit = get_latest_commit_sha(
    "shuheilocale/japanese-personal-name-dataset", "main"
)

_jpersonwiki_path, jpersonwiki_version = get_jpersonwiki_path()
jpersonwiki = pd.read_csv(_jpersonwiki_path, low_memory=False)
jpersonwiki = jpersonwiki[jpersonwiki["is_japanese_name"] == True]
jpersonwiki = jpersonwiki[["gender", "kanji", "hiragana", "birth_year"]]
jpersonwiki = jpersonwiki.rename(columns={"gender": "sex"})
jpersonwiki = jpersonwiki[jpersonwiki["sex"].isin(["M", "F"])]

enamdict.columns = ["sex", "kanji", "hiragana", "romaji"]
enamdict["sex"] = enamdict["sex"].str.upper()
name_origin.columns = ["kanji", "hiragana", "romaji", "sex"]

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  785k  100  785k    0     0  1699k      0 --:--:-- --:--:-- --:--:-- 1700k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 3042k  100 3042k    0     0  6113k      0 --:--:-- --:--:-- --:--:-- 6110k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  514k  100  514k    0     0  1240k      0 --:--:-- --:--:-- --:--:-- 1242k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  342k  100  342k    0     0   947k      0 --:--:-- --:--:-- --:--:--  947k


In [4]:
def preprocess_jpersonwiki(df: pd.DataFrame) -> pd.DataFrame:
    basic_divider = BasicNameDivider()
    divided = df["kanji"].apply(lambda x: basic_divider.divide_name(str(x)))
    hira_parts = df["hiragana"].str.split()
    df["family_kanji"] = divided.apply(lambda x: x.family)
    df["given_kanji"] = divided.apply(lambda x: x.given)
    df["family_hiragana"] = hira_parts.str[0]
    df["given_hiragana"] = hira_parts.str[1]
    return df[
        [
            "sex",
            "given_kanji",
            "given_hiragana",
            "family_kanji",
            "family_hiragana",
            "birth_year",
        ]
    ]


jpersonwiki["birth_year"] = pd.to_numeric(
    jpersonwiki["birth_year"], errors="coerce"
).astype("Int64")
jpersonwiki = preprocess_jpersonwiki(jpersonwiki)

# given nameのみのデータセット
enamdict_out = enamdict.rename(
    columns={"kanji": "given_kanji", "hiragana": "given_hiragana"}
)[["sex", "given_kanji", "given_hiragana"]]
name_origin_out = name_origin.rename(
    columns={"kanji": "given_kanji", "hiragana": "given_hiragana"}
)[["sex", "given_kanji", "given_hiragana"]]

jpersonwiki["dataset"] = "jpersonwiki"
enamdict_out["dataset"] = "enamdict"
name_origin_out["dataset"] = "name_origin"

dataset = pd.concat([jpersonwiki, enamdict_out, name_origin_out], ignore_index=True)
dataset = dataset[
    [
        "dataset",
        "sex",
        "given_kanji",
        "given_hiragana",
        "family_kanji",
        "family_hiragana",
        "birth_year",
    ]
]
os.makedirs("./jnamegender", exist_ok=True)
os.makedirs("./jpersonwiki", exist_ok=True)
dataset.to_csv("./jnamegender/jnamegender.csv", index=False)
jpersonwiki.to_csv("./jpersonwiki/jpersonwiki.csv", index=False)
enamdict_out.to_csv("./jnamegender/enamdict.csv", index=False)
name_origin_out.to_csv("./jnamegender/name_origin.csv", index=False)
!sync

In [5]:
for name, df in [
    ("enamdict", enamdict),
    ("jpersonwiki", jpersonwiki),
    ("name_origin", name_origin),
]:
    print(name)
    print(df.head())

enamdict
  sex kanji hiragana   romaji
0   M   亜喜生      あきお     Akio
1   M   亜喜彦     あきひこ  Akihiko
2   M   亜喜良      あきら    Akira
3   M   亜希人     あきひと  Akihito
4   M   亜希彦     あきひこ  Akihiko
jpersonwiki
  sex given_kanji given_hiragana family_kanji family_hiragana  birth_year  \
0   M          英機            ひでき           東條           とうじょう        1884   
1   M         朝融王          あさあきら          久邇宮              くに        1901   
2   M         由紀夫            ゆきお           三島             みしま        1925   
3   M          正王           もりまさ        梨本宮守             なしもと        1874   
4   M          八郎         へいはちろう         東郷 平            とうごう        1848   

       dataset  
0  jpersonwiki  
1  jpersonwiki  
2  jpersonwiki  
3  jpersonwiki  
4  jpersonwiki  
name_origin
  kanji hiragana romaji sex
0   亜明日      ああす    asu   M
1   亜々人      ああと    ato   M
2     藍       あい     ai   M
3    愛一     あいいち  aichi   M
4    藍一     あいいち  aichi   M


In [6]:
# 再現性のためのメタデータ: 生成時点の行数と、各入力データソースの実際のバージョン/commitを記録する。
# 常にこのセルで最新のdataset/jpersonwikiの状態から再生成し、Kaggleへアップロードする版にも同梱する。
# (このnotebook自体のgit commit hashはKaggle Notebook環境では取得できないため記録しない)
import json
from datetime import datetime, timezone

meta = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "crawler_repo_url": "https://github.com/ren255/jpersondata_crawler",
    "kaggle_dataset_url": "https://www.kaggle.com/datasets/rentoda/jnamegender",
    "total_rows": len(dataset),
    "sources": {
        "jpersonwiki": {
            "kaggle_dataset_url": "https://www.kaggle.com/datasets/rentoda/jpersonwiki",
            "version": jpersonwiki_version,
            "rows": len(jpersonwiki),
        },
        "enamdict": {
            "repo_url": "https://github.com/rgamici/japanese-names",
            "branch": "master",
            "commit": enamdict_source_commit,
            "rows": len(enamdict_out),
        },
        "name_origin": {
            "repo_url": "https://github.com/shuheilocale/japanese-personal-name-dataset",
            "branch": "main",
            "commit": name_origin_source_commit,
            "rows": len(name_origin_out),
        },
    },
}

with open("./jnamegender/meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(meta)

{'generated_at': '2026-11-09T05:57:34.493700+00:00', 'crawler_repo_url': 'https://github.com/ren255/jpersondata_crawler', 'kaggle_dataset_url': 'https://www.kaggle.com/datasets/rentoda/jnamegender', 'total_rows': 409331, 'sources': {'jpersonwiki': {'kaggle_dataset_url': 'https://www.kaggle.com/datasets/rentoda/jpersonwiki', 'version': None, 'rows': 202100}, 'enamdict': {'repo_url': 'https://github.com/rgamici/japanese-names', 'branch': 'master', 'commit': '1d956ba5a7401d1b444e4457cffc851965876b7d', 'rows': 116009}, 'name_origin': {'repo_url': 'https://github.com/shuheilocale/japanese-personal-name-dataset', 'branch': 'main', 'commit': '53251b4207d107fc81a29cd333573d9d1e39ec57', 'rows': 91222}}}


In [7]:
# rentoda/jnamegender へアップロード(jpersonwiki_crawler.ipynbの最後のセルと同じ認証パターン)
import sys
import tempfile
import zipfile
from pathlib import Path

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    KAGGLE_USERNAME = user_secrets.get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY = user_secrets.get_secret("KAGGLE_KEY")
else:
    from dotenv import dotenv_values

    env = dotenv_values(".env")
    KAGGLE_USERNAME = env.get("KAGGLE_USERNAME")
    KAGGLE_KEY = env.get("KAGGLE_KEY")

if not (KAGGLE_USERNAME and KAGGLE_KEY):
    print("Kaggle credentials not found. Skipping dataset upload.")
else:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY

    handle = "rentoda/jnamegender"
    version_notes = (
        f"auto update {datetime.now().strftime('%Y/%m/%d')} "
        f"{len(dataset):,} rows (jpersonwiki v{jpersonwiki_version})"
    )

    with tempfile.TemporaryDirectory() as tmp:
        zip_path = Path(tmp) / "dataset.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write("./jnamegender/jnamegender.csv", "jnamegender.csv")
            zf.write("./jnamegender/meta.json", "meta.json")

        kagglehub.dataset_upload(
            handle=handle,
            local_dataset_dir=str(zip_path.parent),
            version_notes=version_notes,
        )

    print(f"Uploaded new version to {handle} with notes: {version_notes}")

Uploading Dataset https://api.kaggle.com/datasets/rentoda/jnamegender ...
Starting upload for file /tmp/tmpzr7gy6c9/dataset.zip

Uploading: 100%|██████████| 4.18M/4.18M [00:01<00:00, 3.35MB/s]

Upload successful: /tmp/tmpzr7gy6c9/dataset.zip (4MB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/rentoda/jnamegender
Uploaded new version to rentoda/jnamegender with notes: auto update 2026/11/09 409,331 rows (jpersonwiki vNone)
